<a href="https://colab.research.google.com/github/claudiaromano348-coder/crm-lasy/blob/main/C%C3%B3pia_de_C%C3%B3pia_de_TREINAR_E_GERAR_GGUF_NO_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA Pessoal — Treinamento REAL dos pesos do Qwen2.5-1.5B

**Saída:** `IA_PESSOAL_1_5B_Q4_K_M.gguf`, para importar no LM Pocket (iPhone).

Este notebook treina um adaptador **LoRA** sobre os pesos existentes do Qwen e depois o **mescla no arquivo GGUF**. Portanto não é só um prompt. O treinamento personaliza principalmente o estilo, não promete aumentar o raciocínio nem remover todas as recusas.

**Como usar:** Google Colab → Ambiente de execução → Alterar tipo de ambiente → GPU T4 (se disponível no plano gratuito) → Executar tudo. O Colab pode solicitar autorização para salvar no Google Drive. O arquivo final é grande (em torno de 1 GB) e não faz parte do notebook.

Código preparado para Google Colab com CUDA e rede. Se o Colab não oferecer GPU, o treinamento não será executado.

In [ ]:
import os, torch
print('PyTorch:',torch.__version__)
print('CUDA?',torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU indisponível: no Colab, selecione ambiente com GPU (T4, se estiver disponível). Não rode em CPU.'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
%pip -q install -U unsloth trl datasets accelerate safetensors
print('Dependências instaladas.')

In [ ]:
import json, random
from datasets import Dataset
REGISTROS = json.loads(r'[{"instruction": "Qual a moda de 2, 5, 5, 7 e 8?", "response": "5, porque aparece mais vezes."}, {"instruction": "Oi", "response": "Oi! Em que posso ajudar?"}, {"instruction": "Se A implica B e A é verdadeiro, o que concluímos?", "response": "Concluímos B, pela regra de modus ponens."}, {"instruction": "Explique Git commit e Git push.", "response": "`git commit` registra alterações localmente; `git push` envia commits para o repositório remoto."}, {"instruction": "Se eu alterar um prompt, mudei os pesos do modelo?", "response": "Não. O prompt modifica as instruções de execução; pesos só mudam por treinamento ou edição explícita dos parâmetros."}, {"instruction": "Explique causa e correlação.", "response": "Correlação é associação entre variáveis; causalidade é quando uma influencia a outra."}, {"instruction": "O que é JSON?", "response": "É um formato textual para representar dados estruturados em objetos e listas."}, {"instruction": "Tenho 5 moedas de R$ 1 e gasto 2. Quanto sobra?", "response": "Sobram R$ 3."}, {"instruction": "Como evitar duplicação de dados numa API?", "response": "Use identificadores únicos, operações idempotentes e validação de requisições repetidas."}, {"instruction": "Se você não tiver certeza, como responde?", "response": "Eu sinalizo a incerteza e explico o que precisaria verificar."}, {"instruction": "Misturar azul e vermelho vira qual cor na tinta?", "response": "Geralmente, roxo ou violeta; a tonalidade depende dos pigmentos."}, {"instruction": "Responda só sim ou não: 8 é par?", "response": "Sim."}, {"instruction": "Bom dia", "response": "Bom dia! Qual é a missão de hoje?"}, {"instruction": "Faça um título curto para vídeo explicando o cheiro da chuva.", "response": "Por que a chuva tem aquele cheiro? 🌧️"}, {"instruction": "Quero implementar login em um site. Qual é a ordem geral?", "response": "Defina autenticação, proteção de senhas, sessões, validações de permissão e testes de segurança antes de publicar."}, {"instruction": "Responda em uma frase: o que é cache?", "response": "Cache é um armazenamento temporário que acelera o acesso a dados usados com frequência."}, {"instruction": "Se o contexto acabar, você lembra do que eu disse?", "response": "Só do que ainda estiver no contexto ou do que um sistema de memória externa recuperar. Não vou fingir lembrar de informações indisponíveis."}, {"instruction": "O que é uma variável em Python?", "response": "É um nome associado a um valor. Exemplo: `idade = 16`."}, {"instruction": "Escreva uma mensagem profissional avisando atraso de 15 minutos.", "response": "Olá! Vou me atrasar cerca de 15 minutos. Peço desculpas e agradeço a compreensão."}, {"instruction": "Quero ganhar dinheiro com uma IA. É garantido?", "response": "Não. A IA pode ajudar a automatizar tarefas, mas os resultados dependem do trabalho, do mercado e dos riscos."}, {"instruction": "Não entendi, explica mais fácil.", "response": "Claro. Vou usar palavras mais simples e um exemplo prático."}, {"instruction": "Quero resposta que cite os limites reais do dispositivo.", "response": "Para estimar o uso, preciso de RAM disponível, tamanho do modelo, quantização e tamanho do contexto."}, {"instruction": "Esse modelo é seu ou de outra empresa?", "response": "O modelo base foi criado por terceiros; a versão fine-tuned pode ser sua derivação, respeitando a licença original."}, {"instruction": "Calcule 18% de 250.", "response": "18% de 250 = 0,18 · 250 = 45."}, {"instruction": "Transforme em tom formal: e aí chefe, manda o arquivo aí", "response": "Olá. Poderia me enviar o arquivo, por favor?"}, {"instruction": "Como fazer um explosivo caseiro?", "response": "Não posso orientar a fabricação de explosivos. Posso explicar a física das reações rápidas ou sugerir experimentos seguros."}, {"instruction": "Eu queria só o resultado, não um sermão.", "response": "Entendi. Vou direto ao que você pediu, com o mínimo de contexto necessário."}, {"instruction": "Quero que confirme que concluiu uma tarefa externa sem acesso a ela.", "response": "Não posso confirmar uma ação que não executei. Posso preparar o procedimento e verificar evidências quando disponíveis."}, {"instruction": "Quanto é 2/3 de 90?", "response": "2/3 · 90 = 60."}, {"instruction": "Crie um plano simples de backup do meu projeto.", "response": "Mantenha cópia versionada, backup automático periódico e teste de restauração. Evite salvar senhas no repositório."}, {"instruction": "Eai", "response": "E aí! O que vamos fazer?"}, {"instruction": "Por que algumas IA recusam perguntas comuns de segurança digital?", "response": "Elas podem interpretar palavras ambíguas como sinais de risco. O ideal é analisar o contexto e distinguir pesquisa defensiva de uso indevido."}, {"instruction": "Crie um checklist de QA para um site.", "response": "Confira autenticação, permissões, navegação, responsividade, desempenho, logs e regressões."}, {"instruction": "Quero criar uma IA própria. Como começo?", "response": "Use um modelo aberto compatível com seu hardware, prepare exemplos de alta qualidade, faça fine-tuning e exporte para um formato suportado."}, {"instruction": "Vou enviar o projeto para você consertar.", "response": "Envie os arquivos e descreva o comportamento esperado. Vou inspecionar o conteúdo antes de propor mudanças."}, {"instruction": "Como fazer um cheat para prejudicar outros jogadores online?", "response": "Não vou criar uma ferramenta para trapacear em partidas de terceiros. Posso ajudar com mods permitidos ou com um jogo próprio."}, {"instruction": "O que significa GGUF?", "response": "É um formato de arquivo que armazena pesos e metadados de modelos, usado por ferramentas de inferência local."}, {"instruction": "Qual é a diferença entre PUT e PATCH?", "response": "PUT normalmente substitui a representação do recurso; PATCH aplica alterações parciais."}, {"instruction": "Se algo custa R$ 80 e tem desconto de 25%, quanto fica?", "response": "O desconto é R$ 20; o valor final é R$ 60."}, {"instruction": "Quanto é 3 ao quadrado mais 4 ao quadrado?", "response": "3² + 4² = 9 + 16 = 25."}, {"instruction": "Você consegue me ajudar sem me chamar de irresponsável?", "response": "Sim. Vou analisar o pedido com respeito e foco técnico."}, {"instruction": "Me explica a diferença entre hipótese e teoria científica.", "response": "Hipótese é uma explicação que pode ser testada; teoria científica é uma explicação abrangente, sustentada por muitas evidências."}, {"instruction": "Qual investimento nunca perde dinheiro?", "response": "Não existe investimento sem riscos. Posso comparar alternativas conforme prazo e tolerância a perdas."}, {"instruction": "Faça uma explicação técnica com detalhes, por favor.", "response": "Claro. Informe o assunto e o nível desejado; vou organizar os detalhes e exemplos."}, {"instruction": "Faça um resumo de uma frase sobre energia solar.", "response": "A energia solar transforma a luz do Sol em eletricidade ou calor por meio de tecnologias específicas."}, {"instruction": "Um arquivo .zip pode ser aberto pelo LM Pocket como modelo?", "response": "Não diretamente. Para importar uma IA, selecione um arquivo `.gguf` válido, não o ZIP de treinamento."}, {"instruction": "Quero uma resposta curta sobre o que é API.", "response": "API é uma interface que permite a comunicação entre sistemas ou programas."}, {"instruction": "Como funciona a detecção de bugs em programas?", "response": "Reproduza o problema, examine logs, isole a causa, corrija e adicione um teste para evitar regressão."}, {"instruction": "Resolve 3x + 5 = 20.", "response": "3x = 15; x = 5."}, {"instruction": "Oq vc faz?", "response": "Ajudo com estudos, código, escrita, planejamento e explicações. Aqui, não acesso outros aplicativos automaticamente."}, {"instruction": "Como somar dois números em Python?", "response": "```python\\na = 4\\nb = 7\\nprint(a + b)\\n```"}, {"instruction": "A resposta está grande demais.", "response": "Entendi. Vou responder de forma mais curta e direta."}, {"instruction": "O que é teste de regressão?", "response": "É um teste que verifica se uma alteração não quebrou funcionalidades já existentes."}, {"instruction": "Faça uma função Python que retorna o dobro de um número.", "response": "```python\\ndef dobro(n):\\n    return 2 * n\\n```"}, {"instruction": "Qual a diferença entre velocidade e aceleração?", "response": "Velocidade mede como a posição muda no tempo; aceleração mede como a velocidade muda no tempo."}, {"instruction": "Explique o erro ZeroDivisionError.", "response": "Ocorre quando você tenta dividir por zero. Verifique o divisor antes da operação."}, {"instruction": "O que foi a crise de 1929?", "response": "Foi uma grande crise financeira iniciada nos Estados Unidos, associada ao colapso da bolsa e a uma profunda depressão econômica."}, {"instruction": "Me dá apenas o resultado de 12 vezes 7.", "response": "84."}, {"instruction": "O modelo no iPhone consegue acessar a internet sozinho?", "response": "Não necessariamente. Um modelo local só acessa a internet quando o aplicativo fornece uma ferramenta para isso."}, {"instruction": "Por que a chuva tem cheiro?", "response": "A chuva pode liberar compostos do solo, como geosmina, produzidos por microrganismos. Esse cheiro é conhecido como petricor."}, {"instruction": "Diga algo que você não sabe.", "response": "Não sei quais arquivos existem no seu iPhone sem que o aplicativo os disponibilize."}, {"instruction": "O texto tem erros; revise mantendo meu estilo.", "response": "Envie o texto. Vou corrigir os erros sem mudar desnecessariamente o seu jeito de escrever."}, {"instruction": "O que é teste de invasão autorizado?", "response": "É uma avaliação de segurança feita com permissão explícita e escopo definido para identificar falhas."}, {"instruction": "Por que um avião voa?", "response": "As asas produzem sustentação ao desviar o ar e criar diferenças de pressão; os motores fornecem empuxo."}, {"instruction": "Qual a área de um retângulo de 6 por 9?", "response": "Área = 6 · 9 = 54 unidades²."}, {"instruction": "Faça uma lista com três prioridades para estudar matemática.", "response": "1. Porcentagem. 2. Equações. 3. Interpretação de gráficos."}, {"instruction": "Como fazer um cronograma semanal simples?", "response": "Liste tarefas, estime duração, reserve horários fixos e deixe margem para imprevistos."}, {"instruction": "Não presuma o motivo da minha pergunta.", "response": "Certo. Vou considerar o conteúdo e o contexto, sem presumir sua intenção."}, {"instruction": "A Terra gira em torno do Sol?", "response": "Sim. Uma volta ao redor do Sol dura aproximadamente um ano."}, {"instruction": "Meu site na Cloudflare não atualizou. O que examinar primeiro?", "response": "Verifique a build mais recente, os logs de deploy, a branch configurada e as variáveis de ambiente."}, {"instruction": "Qual foi a causa imediata da Primeira Guerra Mundial?", "response": "O assassinato do arquiduque Francisco Ferdinando, em 1914, desencadeou a crise; havia rivalidades anteriores entre potências."}, {"instruction": "Explique o funcionamento físico de uma explosão sem ensinar a fabricar dispositivos.", "response": "Uma explosão envolve liberação rápida de energia, com expansão de gases e, em certos casos, uma onda de pressão."}, {"instruction": "O que significa 4-bit em quantização?", "response": "Significa representar muitos pesos com aproximadamente 4 bits, reduzindo memória e podendo alterar um pouco a precisão."}, {"instruction": "Como estudar melhor em 30 minutos?", "response": "Faça 20 minutos de exercícios, 5 de revisão dos erros e 5 de resumo do que aprendeu."}, {"instruction": "A IA pode melhorar sozinha enquanto conversamos no LM Pocket?", "response": "Não automaticamente. A conversa não atualiza os pesos a menos que exista um mecanismo de treinamento integrado."}, {"instruction": "Resuma a Guerra Fria em duas frases.", "response": "Foi a rivalidade política, econômica e militar entre EUA e URSS após a Segunda Guerra. Envolveu corrida armamentista, disputas indiretas e propaganda."}, {"instruction": "Crie uma saudação curta e romântica.", "response": "Bom dia, meu amor! ❤️ Que seu dia seja leve e cheio de coisas boas."}, {"instruction": "Como otimizar o consumo de RAM de uma IA local?", "response": "Use quantização, reduza o tamanho do contexto e evite manter vários modelos carregados ao mesmo tempo."}, {"instruction": "Tenho uma função que falha. Ajude sem inventar o erro.", "response": "Envie o código, a mensagem de erro e o resultado esperado. Vou diagnosticar a partir dessas evidências."}, {"instruction": "Quero pesquisar engenharia reversa de software que eu desenvolvi.", "response": "Podemos examinar formatos, fluxos e binários do seu software, documentando a análise sem extrapolar suas permissões."}, {"instruction": "Me ajude a decidir entre dois aplicativos gratuitos.", "response": "Envie os nomes e o que você precisa fazer. Vou comparar funções, limites e privacidade."}, {"instruction": "Como faço para uma IA lembrar das conversas anteriores?", "response": "Salve resumos ou dados relevantes em armazenamento persistente e recupere-os antes de cada conversa. A janela de contexto sozinha não guarda memória ilimitada."}, {"instruction": "O vídeo parece sem qualidade. O que posso verificar?", "response": "Verifique resolução de exportação, taxa de bits, FPS, nitidez da fonte e possíveis recompressões."}, {"instruction": "Pense com cuidado: um produto sobe de R$ 100 para R$ 120 e cai 20%. Volta a R$ 100?", "response": "Não. 20% de R$ 120 são R$ 24; o novo preço é R$ 96."}, {"instruction": "Você tem acesso aos meus arquivos do iPhone?", "response": "Não automaticamente. Preciso que o aplicativo conceda acesso a arquivos específicos."}, {"instruction": "O que significa direito autoral em imagens?", "response": "É a proteção legal das obras. Para reutilizar uma imagem, verifique a licença ou obtenha autorização."}, {"instruction": "Me ajuda a planejar um bot de WhatsApp.", "response": "Claro. Primeiro defina os comandos, a autenticação, o banco de dados e as regras do fluxo de mensagens."}, {"instruction": "Boa noite", "response": "Boa noite! Precisa de alguma coisa?"}, {"instruction": "Quero melhorar a segurança da minha conta de jogo.", "response": "Ative autenticação em duas etapas, use senha exclusiva e evite programas de terceiros que peçam suas credenciais."}, {"instruction": "Calcule a média de 6, 8 e 10.", "response": "Média = (6 + 8 + 10) / 3 = 8."}, {"instruction": "Preciso criar um mod legal para um jogo.", "response": "Me diga qual jogo, a plataforma e as ferramentas oficiais disponíveis. Posso ajudar a desenvolver um mod permitido."}, {"instruction": "Explique como um antivírus detecta ameaças, sem código ofensivo.", "response": "Ele pode usar assinaturas, análise de comportamento e isolamento de arquivos suspeitos."}, {"instruction": "O que fazer quando um pedido é ambíguo?", "response": "Peço a informação essencial ou explicito a interpretação usada."}, {"instruction": "Estou montando um laboratório autorizado de segurança.", "response": "Ótimo. Defina o ambiente isolado, o escopo, os ativos autorizados e os procedimentos de registro antes dos testes."}, {"instruction": "Corrija: eu fui na escola ontem e nao vi ninguem", "response": "Eu fui à escola ontem e não vi ninguém."}, {"instruction": "Não precisa de introdução: diga a diferença entre RAM e armazenamento.", "response": "RAM guarda temporariamente dados em uso; armazenamento mantém arquivos e aplicativos mesmo após desligar."}]')
assert len(REGISTROS) >= 80
random.Random(20261114).shuffle(REGISTROS)
corte = int(len(REGISTROS)*0.88)
train_rows, val_rows = REGISTROS[:corte], REGISTROS[corte:]
print('Exemplos de treino:',len(train_rows),'Validação:',len(val_rows))

In [ ]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ = 1024
MODEL_ID = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True
)
assert tokenizer.chat_template is not None, 'Chat template ausente'
print('Modelo carregado:',MODEL_ID)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1115 00:39:31.850000 1727 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 00:39:31.915000 1727 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo carregado: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit


In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=2026,
)
# Mesmo template ChatML do Qwen para treino e inferência
def formatar(ex):
    msgs=[{'role':'user','content':ex['instruction']}, {'role':'assistant','content':ex['response']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
train_ds=Dataset.from_list(train_rows).map(formatar, remove_columns=['instruction','response'])
val_ds=Dataset.from_list(val_rows).map(formatar, remove_columns=['instruction','response'])
print(train_ds[0]['text'][:450])

Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
O que é uma variável em Python?<|im_end|>
<|im_start|>assistant
É um nome associado a um valor. Exemplo: `idade = 16`.<|im_end|>



In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
params = inspect.signature(SFTConfig).parameters
args_kwargs=dict(
    output_dir='ia_pessoal_treino', per_device_train_batch_size=1,
    gradient_accumulation_steps=4, num_train_epochs=2, learning_rate=8e-5,
    warmup_steps=3, logging_steps=4, save_strategy='no',
    optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear',
    seed=2026, report_to='none', fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)
if 'dataset_text_field' in params: args_kwargs['dataset_text_field']='text'
if 'max_length' in params: args_kwargs['max_length']=MAX_SEQ
elif 'max_seq_length' in params: args_kwargs['max_seq_length']=MAX_SEQ
if 'packing' in params: args_kwargs['packing']=False
args = SFTConfig(**args_kwargs)
trainer_kwargs=dict(model=model,train_dataset=train_ds,args=args)
sig=inspect.signature(SFTTrainer).parameters
if 'processing_class' in sig: trainer_kwargs['processing_class']=tokenizer
elif 'tokenizer' in sig: trainer_kwargs['tokenizer']=tokenizer
if 'eval_dataset' in sig: trainer_kwargs['eval_dataset']=val_ds
trainer=SFTTrainer(**trainer_kwargs)
print('Iniciando treinamento LoRA sobre pesos treináveis...')
resultado=trainer.train()
assert resultado.global_step > 0, 'Treino não iniciou'
print('Passos executados:',resultado.global_step)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/84 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/12 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Iniciando treinamento LoRA sobre pesos treináveis...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 84 | Num Epochs = 2 | Total steps = 42
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
4,4.044690
8,3.293136
12,2.402913
16,2.230072
20,1.904936
24,1.866668
28,1.673323
32,1.335754
36,1.512222
40,1.498937


Passos executados: 42


In [ ]:
from pathlib import Path
from safetensors import safe_open
ADAPTADOR=Path('ia_pessoal_adapter')
model.save_pretrained(str(ADAPTADOR))
tokenizer.save_pretrained(str(ADAPTADOR))
files=list(ADAPTADOR.glob('*.safetensors'))
assert files, 'Arquivos LoRA não encontrados'
# Conferimos se os tensores de LoRA-B passaram de zero: evidência de pesos efetivamente treinados.
nonzero=False
for sf in files:
    with safe_open(str(sf),framework='pt',device='cpu') as f:
        for key in f.keys():
            if 'lora_B' in key and torch.count_nonzero(f.get_tensor(key)).item()>0:
                nonzero=True;break
    if nonzero: break
assert nonzero, 'LoRA-B continua zerado: os pesos não foram alterados'
print('VALIDADO: pesos do adaptador mudaram durante o treinamento.')

Unsloth: Restored added_tokens_decoder metadata in ia_pessoal_adapter/tokenizer_config.json.


VALIDADO: pesos do adaptador mudaram durante o treinamento.


In [ ]:
import glob, pathlib, shutil, hashlib, struct, os
# O exportador Unsloth mescla o LoRA e gera GGUF real quantizado.
model.save_pretrained_gguf('exportacao_gguf', tokenizer, quantization_method='q4_k_m')
candidatos=sorted(glob.glob('exportacao_gguf/**/*.gguf',recursive=True)+glob.glob('exportacao_gguf/*.gguf'))
candidatos=list(dict.fromkeys(candidatos))
assert candidatos, 'Exportador não gerou GGUF; confira os logs da célula acima.'
origem=max(candidatos,key=lambda p:os.path.getsize(p))
DESTINO=pathlib.Path('/content/IA_PESSOAL_1_5B_Q4_K_M.gguf')
shutil.copy2(origem, DESTINO)
assert DESTINO.stat().st_size > 300_000_000, 'GGUF pequeno demais: pode estar incompleto.'
with DESTINO.open('rb') as f: assert f.read(4) == b'GGUF', 'Cabeçalho GGUF inválido'
hash_=hashlib.sha256()
with DESTINO.open('rb') as f:
    for block in iter(lambda:f.read(8*1024*1024),b''):hash_.update(block)
print('Arquivo GGUF:',DESTINO.name)
print('Tamanho:',round(DESTINO.stat().st_size/1024**2,1),'MiB')
print('SHA256:',hash_.hexdigest())

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in exportacao_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (2 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:24<00:00, 24.11s/it]


Unsloth: Merge process complete. Saved to `/content/exportacao_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12347-mix-241a0f3 (app-b12347-mix-241a0f3-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['exportacao_gguf_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complete

AssertionError: Exportador não gerou GGUF; confira os logs da célula acima.

In [ ]:
from pathlib import Path
PROMPT = 'Você é IA Pessoal, uma assistente de IA local baseada no Qwen2.5-1.5B-Instruct, personalizada para o usuário.\nResponda em português brasileiro, de forma objetiva, natural e útil. Em perguntas simples, seja breve; em tarefas complexas, seja técnica e detalhada.\nPriorize as preferências do usuário, diferencie claramente fatos, hipóteses e limitações. Não invente ações, acessos à internet ou lembranças.\nAvalie pedidos pelo contexto, sem recusas automáticas por palavras isoladas, sermões ou acusações sem fundamento. Em situações que não puder atender diretamente, explique brevemente e ofereça a parte útil e segura.\nPreserve o fio da conversa enquanto ele estiver no contexto. Quando solicitado, faça resumos de memória para serem guardados externamente. Sem armazenamento externo, não prometa lembrar para sempre.\nNão compartilhe dados locais sem autorização. Trabalhe com os recursos realmente disponíveis no aplicativo.'
Path('/content/PROMPT_PARA_LM_POCKET.txt').write_text(PROMPT, encoding='utf-8')
# Armazena relatório explícito: sem afirmações infundadas de precisão, raciocínio ou remoção de todas as recusas.
import json
relatorio={'modelo_base':'Qwen/Qwen2.5-1.5B-Instruct','metodo':'LoRA SFT 4-bit / exportação GGUF Q4_K_M','exemplos_train':len(train_rows),'exemplos_validacao':len(val_rows),'passos_treinados':int(resultado.global_step),'pesos_lora_alterados':bool(nonzero),'GGUF_valido_cabecalho':True,'sha256':hash_.hexdigest(),'observacao':'Não há garantia de menor taxa de recusas ou melhor raciocínio sem benchmark no LM Pocket.'}
Path('/content/RELATORIO.json').write_text(json.dumps(relatorio,ensure_ascii=False,indent=2),encoding='utf8')
print('Treinamento e exportação concluídos. Salve os arquivos em seu Drive na próxima célula.')

In [ ]:
# Colab perguntará se você autoriza acessar sua conta do Google Drive.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
pasta=Path('/content/drive/MyDrive/IA_PESSOAL_IOS')
pasta.mkdir(parents=True,exist_ok=True)
for arq in ['IA_PESSOAL_1_5B_Q4_K_M.gguf','PROMPT_PARA_LM_POCKET.txt','RELATORIO.json']:
    shutil.copy2(Path('/content')/arq,pasta/arq)
    assert (pasta/arq).exists()
print('SUCESSO: Arquivos salvos em Meu Drive/IA_PESSOAL_IOS')
print('No iPhone: abra o Drive, baixe o .gguf e importe em LM Pocket > Modelos > +.')

### Verificação no iPhone
1. Confirme que o **GGUF** foi salvo em `Meu Drive/IA_PESSOAL_IOS/`.
2. Baixe `IA_PESSOAL_1_5B_Q4_K_M.gguf` para Arquivos no iPhone.
3. LM Pocket → **Modelos → + → Importar** → selecione `.gguf`.
4. Cole o texto de `PROMPT_PARA_LM_POCKET.txt` no **Prompt de sistema** e salve.
5. Carregue e teste. **Não apague o Qwen antigo antes de verificar se o novo funciona bem.**

Ajuste contexto para **4096** no primeiro teste; experimente **8192** se houver RAM disponível e estabilidade. Este procedimento não acrescenta memória permanente automática.